In [4]:
import cobra
import os
import pickle
import requests
import re

cow= cobra.io.read_sbml_model('/data/user_home/dlwnsrb/cow_reconstruction/change_gene_name/change_gene_name_cow.xml')

with open('/data/user_home/dlwnsrb/cow_reconstruction/data/gene_reaction.pickle', 'rb') as f:
    gene_reaction= pickle.load(f)

In [5]:
#gene reaction add draft
kegg_reaction_id_original= set()
kegg_metabolite_id_original= set()

for reaction in cow.reactions:
    if 'kegg.reaction' in reaction.annotation:
        kegg_reaction_id_original.add(reaction.annotation['kegg.reaction'])

for metabolite in cow.metabolites:
    if 'kegg.compound' in metabolite.annotation:
        kegg_metabolite_id_original.add(metabolite.annotation['kegg.compound'])


gene_add= set()
for gene, reaction_list in gene_reaction.items():
    if not gene in [gene.id for gene in cow.genes]:
        gene_add.add(gene)
                
reaction_add= set()
for gene, reaction_list in gene_reaction.items():
    if not gene in [gene.id for gene in cow.genes]:
        for reaction in reaction_list:
            if reaction not in kegg_reaction_id_original:
                reaction_add.add(reaction)

In [6]:
#identify reaction_overlap
reaction_overlap= dict()

for reaction in reaction_add:
    url= 'https://rest.kegg.jp/get/'+reaction
    requestData = requests.get(url).text
    for data in requestData.split('\n'):
        if 'REMARK      Same as:' in data:
            reaction_overlap[reaction]= data.split('REMARK      Same as: ')[1].split(' ')
    if not reaction in reaction_overlap:
        reaction_overlap[reaction]= []


In [16]:
#reaction add할것, 겹치는것 조사
pattern = r'[C,G]([0-9]*)'
reaction_add_final= set()
reaction_overlap_final= dict()
for reaction, overlap_list in reaction_overlap.items():
    not_valid_reaction_criteria= False
    #about reaction(key)
    url= 'https://rest.kegg.jp/get/'+reaction
    requestData = requests.get(url).text
    count_reaction=0
    for data in requestData.split('\n'):
        if 'EQUATION' in data:
            reactants, products= data.split('EQUATION    ')[1].split(' <=> ')
            reactants_list= reactants.split(' + ')
            products_list= products.split(' + ')

            #reactant와 product가 같이 나오면 없애버리기
            reactant_name_list=list()
            product_name_list=list()
            for reactant in reactants_list:
                a= re.search(pattern,reactant)
                reactant_name= a.group()
                reactant_name_list.append(reactant_name)
            for product in products_list:
                a= re.search(pattern,product)
                product_name= a.group()
                product_name_list.append(product_name)
            for reactant_name in reactant_name_list:
                if reactant_name in product_name_list:
                    not_valid_reaction_criteria= True
            if not_valid_reaction_criteria:
                break
            total_metabolite_list= reactants_list+ products_list
            for metabolite in total_metabolite_list:
                a= re.search(pattern,metabolite)
                string= a.group()
                if string in kegg_metabolite_id_original:
                    count_reaction+=1
                    
    if not_valid_reaction_criteria:
        continue
        
    if len(overlap_list)==0:
        reaction_add_final.add(reaction)
        
        
    else:
        for reaction_ in overlap_list:
            #kegg_reaction_id_original에 있으면 추가할 필요없이 reaction_overlap_final에서 언급만 해주면 됨
            if reaction_ in kegg_reaction_id_original:
                reaction_overlap_final[reaction]= reaction_
                break
            else:
                count_reaction_= 0
                #about reaction_(value)
                url= 'https://rest.kegg.jp/get/'+reaction_
                requestData = requests.get(url).text
                for data in requestData.split('\n'):
                    if 'EQUATION' in data:
                        reactants, products= data.split('EQUATION    ')[1].split(' <=> ')
                        reactants_list= reactants.split(' + ')
                        products_list= products.split(' + ')
                        total_metabolite_list= reactants_list+ products_list
                        for metabolite in total_metabolite_list:
                            a= re.search(pattern,metabolite)
                            string= a.group()
                            if string in kegg_metabolite_id_original:
                                count_reaction_+=1
                #만약 reaction_이 reaction_overlap에 없고, reaction이 add되는 상황이면 reaction_add_final에 넣을 reaction만 추가. 다른 reaction은 삭제하면 됨
                #그러나 만약 reaction_이 reaction_overlap에 있으면, reaction_overlap_final도 고려해야됨
                if reaction_ not in reaction_overlap:
                    if count_reaction < count_reaction_:
                        reaction_add_final.add(reaction_)
                        reaction_overlap_final[reaction]= reaction_
                    else:
                        reaction_add_final.add(reaction)
                else:
                    if count_reaction < count_reaction_:
                        reaction_add_final.add(reaction_)
                        reaction_overlap_final[reaction]= reaction_
                    else:
                        reaction_add_final.add(reaction)
                        reaction_overlap_final[reaction_]= reaction

In [17]:
pattern = r'[C,G]([0-9]*)'

reaction_add_final_equation= dict()
metabolite_set= set()
addition_metabolite_draft= list()
metabolite_match= dict()

for reaction in reaction_add_final:
    url= 'https://rest.kegg.jp/get/'+reaction
    requestData = requests.get(url).text
    for data in requestData.split('\n'):
        if 'EQUATION' in data:
            reactants, products= data.split('EQUATION    ')[1].split(' <=> ')
            reactants_list= reactants.split(' + ')
            products_list= products.split(' + ')
            reaction_add_final_equation[reaction]= [reactants_list, products_list]
            for reactant in reactants_list:
                a= re.search(pattern,reactant)
                string= a.group()
                metabolite_set.add(string)
            for product in products_list:
                a= re.search(pattern,product)
                string= a.group()
                metabolite_set.add(string)
                
for metabolite in metabolite_set:
    if not metabolite in kegg_metabolite_id_original:
        addition_metabolite_draft.append(metabolite)

for metabolite in addition_metabolite_draft:
    criteria=False
    url= 'https://rest.kegg.jp/get/'+metabolite
    requestData = requests.get(url).text
    metabolites= False
    for data in requestData.split('\n'):
        if 'REMARK' in data and 'Same as' in data:
            metabolite_string= data.split('REMARK      Same as: ')[1]
            metabolites= metabolite_string.split(' ')
    if metabolites:
        metabolite_match[metabolite]= metabolites
    else:
        metabolite_match[metabolite]=[]

In [18]:
metabolite_add_final= set()
metabolite_overlap_final= dict()
metabolite_add_final_formula= dict()

for metabolite, overlap_list in metabolite_match.items():
    if len(overlap_list)==0:
        metabolite_formula=False
        metabolite_add_final.add(metabolite)
        url= 'https://rest.kegg.jp/get/'+metabolite
        requestData = requests.get(url).text
        for data in requestData.split('\n'):
            if 'FORMULA' in data:
                metabolite_formula= data.split('FORMULA     ')[1]
                metabolite_add_final_formula[metabolite]= metabolite_formula
        if not metabolite_formula:
            metabolite_add_final_formula[metabolite]=''
    if not len(overlap_list) == 0:
        metabolite_formula= False
        metabolite_formula_= False
        for metabolite_ in overlap_list:
            #Drug annotation은 일단 제외
            if 'D' in metabolite_ and len(overlap_list) !=1 :
                continue
            #kegg_metabolite_id_original에 있으면 추가할 필요없이 metabolite_overlap_final에서 언급만 해주면 됨
            if metabolite_ in kegg_metabolite_id_original:
                metabolite_overlap_final[metabolite]= metabolite_
                break
            else:
                #about metabolite(key)
                url= 'https://rest.kegg.jp/get/'+metabolite
                requestData = requests.get(url).text
                for data in requestData.split('\n'):
                    if 'FORMULA' in data:
                        metabolite_formula= data.split('FORMULA     ')[1]
                        metabolite_add_final_formula[metabolite]= metabolite_formula
                        metabolite_add_final.add(metabolite)
                if metabolite_formula:
                    break
                #about reaction_(value)
                url= 'https://rest.kegg.jp/get/'+metabolite_
                requestData = requests.get(url).text
                for data in requestData.split('\n'):
                    if 'FORMULA' in data:
                        metabolite_formula_= data.split('FORMULA     ')[1]
                        metabolite_add_final_formula[metabolite_]= metabolite_formula_
                        metabolite_add_final.add(metabolite_)
                        metabolite_overlap_final[metabolite]= metabolite_

In [35]:
cow_dict= dict()
for metabolite in cow.metabolites:
    if 'kegg.compound' in metabolite.annotation:
        cow_dict[metabolite.annotation['kegg.compound']]= metabolite.formula

In [101]:
#n 이 있는 formula change
metabolite_add_final_formula['C00369']= 'C30H54O27'
metabolite_add_final_formula['C20265']= 'C21H24O3'
metabolite_add_final_formula['C20796']= 'C22H34N5O19P3R2'
metabolite_add_final_formula['C01935']= 'C12H22O11'
metabolite_add_final_formula['C20751']= 'C25H37N6O23P3R2'
metabolite_add_final_formula['C00828']= 'C21H24O2'
metabolite_add_final_formula['C17323']= 'C22H34N3O22P3SR2'
metabolite_add_final_formula['C20459']= 'C20H30N5O19P3R2'
metabolite_add_final_formula['C20648']= 'C20H30N5O19P3R2'
metabolite_add_final_formula['C06113']= 'C4H5NO3R'
metabolite_add_final_formula['C04157']= 'C6H7N5OR'
metabolite_add_final_formula['C06112']= 'C5H6NO4R'
metabolite_add_final_formula['C02843']= 'C24H35N7O17P3SR'
metabolite_add_final_formula['C20863']= 'C15H21O19P3R3'
metabolite_add_final_formula['C20458']= 'C20H29N4O20P3R2'
metabolite_add_final_formula['C17324']= 'C20H30N5O19P3R2'
metabolite_add_final_formula['C03547']= 'C2H4O3'
metabolite_add_final_formula['C20752']= 'C26H39N6O23P3SR2'
metabolite_add_final_formula['C20935']= 'C20H23N10O19P4'
metabolite_add_final_formula['C17322']= 'C19H29N2O20P3SR2'

In [182]:
#formula가 없는 metabolite 기입
#완전한 formula가 나오는 metabolite가 아니면 최소단위 X로 formula 씀
metabolite_add_final_formula['G13094']= 'C5H13O7PX'
metabolite_add_final_formula['G13027']= 'X'
metabolite_add_final_formula['C15603']= 'X'
metabolite_add_final_formula['G10511']= 'C79H136N4O47R'
metabolite_add_final_formula['C00030']= 'H2X'
metabolite_add_final_formula['G03532']= 'X'
metabolite_add_final_formula['G13091']= 'X'
metabolite_add_final_formula['G13095']= 'C10H22O11PX'
metabolite_add_final_formula['C21758']= 'R'
metabolite_add_final_formula['G00009']= 'C88H148N2O70X'
metabolite_add_final_formula['C19645']= 'X'
metabolite_add_final_formula['G13043']= 'C8H13NO5X'
metabolite_add_final_formula['C03024']= 'H2X'
metabolite_add_final_formula['G12336']= 'C8H13NO8SX'
metabolite_add_final_formula['G10913']= 'C5H9O4X'
metabolite_add_final_formula['G13052']= 'C6H12O4X'
metabolite_add_final_formula['G13053']= 'C14H26NO9X'
metabolite_add_final_formula['C00028']= 'X'
metabolite_add_final_formula['C15602']= 'X'
metabolite_add_final_formula['G13087']= 'C8H14NO5X'
metabolite_add_final_formula['C17023']= 'S'
metabolite_add_final_formula['G13092']= 'O3PX'
metabolite_add_final_formula['C19646']= 'C6H8O6X'
metabolite_add_final_formula['G13093']= 'X'
metabolite_add_final_formula['G00872']= 'X'
metabolite_add_final_formula['C03161']= 'X'

In [183]:
# 계수가 n인 equation 수정
reaction_add_final_equation['R10910']= [['2 C00002'], ['C20935', 'C00013']]

In [189]:
import pickle
with open('metabolite_formula.pickle', 'wb') as f:
    pickle.dump(metabolite_add_final_formula,f)
    
with open('metabolite_add.pickle', 'wb') as f:
    pickle.dump(metabolite_add_final,f)

with open('metabolite_overlap.pickle', 'wb') as f:
    pickle.dump(metabolite_overlap_final,f)

with open('reaction_add.pickle', 'wb') as f:
    pickle.dump(reaction_add_final_equation,f)
    
with open('reaction_overlap.pickle', 'wb') as f:
    pickle.dump(reaction_overlap_final,f)